In [99]:
import re
from enum import unique
from typing import Sequence, Optional

ISO_DATE        = re.compile(r'^(19|20)\d{2}-[01]\d-[0-3]\d$')
SLASH_DATE      = re.compile(r'^[0-3]\d/[01]\d/(19|20)\d{2}$')
VACC_DATE       = re.compile(r'^\d{6}_\w{7,8}$')              # 240119_46576392
EXCEL_SERIAL    = re.compile(r'^\d{5,6}(\.0)?$')
HEX_ID          = re.compile(r'^[0-9a-f]{6,64}$', re.IGNORECASE)
GENOMIC_ID      = re.compile(r'^EPI_ISL_\d{6,9}$', re.IGNORECASE)
AGE_PAT         = re.compile(r'^(0?\d{1,2}|1[01]\d|120)(\.0)?$')

DATE_RE = re.compile(
    r'^(?:'
    # ISO dates: YYYY-MM-DD, years 1900–2099, months 01–12, days 01–31
    r'(?:19|20)\d{2}-(?:0[1-9]|1[0-2])-(?:0[1-9]|[12]\d|3[01])'
    r'|'
    # Slashed dates: DD/MM/YYYY, days 01–31, months 01–12, years 1900–2099
    r'(?:0[1-9]|[12]\d|3[01])/(?:0[1-9]|1[0-2])/(?:19|20)\d{2}'
    r'|'
    # Vaccine codes: six digits, underscore, 7–8 alnum chars
    r'\d{6}_[A-Za-z0-9]{7,8}'
    r'|'
    # Excel serials: 5–6 digits, optional “.0”
    r'\d{5,6}(?:\.0)?'
    r')$'
)

GENERAL_ID = re.compile(
    r'^(?=.*[A-Za-z])(?=.*\d)[A-Za-z0-9.-]{3,30}$',
    re.VERBOSE
)


YES_NO          = re.compile(
    r'^(yes|no|oui|non|unknown|inconnu|y|n|0\.0|1\.0)$',
    re.IGNORECASE
)
GENDER_PAT      = re.compile(
    r'^(male|female|m|f|other)$',
    re.IGNORECASE
)
OUTCOME_PAT     = re.compile(
    r'^(recovered|died|alive|deceased|discharged.*|not recovered|other)$',
    re.IGNORECASE
)
CASE_STATUS_PAT = re.compile(
    r'^(confirme?|prob|disc|suspected case|invalidée|confirmed case)$',
    re.IGNORECASE
)

SYM_DELIMS      = re.compile(r'[|;/]')   # just to token-split if you need

# comma-separated list of roles/persons
CONTACT_COMMENT_PAT = re.compile(
    r"^[A-Z][a-z'-]+ [A-Z][a-z'-]+"
    r"(?:, ?[A-Z][a-z'-]+ [A-Z][a-z'-]+)*$"
)

# confirmation method options
CONFIRMATION_METHOD_PAT = re.compile(
    r'^(?:clinical signs and symptoms only|laboratory confirmed|other|samples)'
    r'(?:,(?:clinical signs and symptoms only|laboratory confirmed|other|samples))*$',
    re.IGNORECASE
)

# crude pipe-delimited contact_animal entries
CONTACT_ANIMAL_PAT = re.compile(r'.*\|.*')

# comma-separated list of occupations (allow slash, hyphens, semicolons)
# strict list of known occupations, case‐insensitive
titles = [
    "doctor","nurse","teacher","driver","farmer","student","engineer",
    "chef","laborer","police officer","police","soldier","technician",
    "manager","cleaner","waiter"
]

# build an “alternation” of those titles, escaping spaces
escaped = [re.escape(t) for t in titles]
group   = "|".join(escaped)

OCCUPATION_PAT = re.compile(
    r'(?i)^'                   # inline case-insensitive, at the very start
    r'(?:' + group + r')'      # one of the known titles
                     r'(?:'                     # optional repeat...
                     r'(?:,\s*|;\s*|/\s*)'    #   separator
                     r'(?:' + group + r')'    #   another title
                                      r')*'                      # zero or more
                                      r'$'                       # end of string
)

LOCATION_PAT = re.compile(
    r"^[A-Z][A-Za-zÀ-ÿ' .-]{3,}(?:, ?[A-Z][A-Za-z]{2,})?$"
)

# known pathogens
PATHOGEN_PAT = re.compile(r'^(?:novhep|cholera)$', re.IGNORECASE)

# generic comma-separated source identifiers (allow underscores, slashes)
SOURCE_PAT = re.compile(
    r'^(?:[A-Z]{2,5}\d{2,}|https?://)', re.IGNORECASE
)
# ------------------------------------------------------------------ #
# RULES  label → list[pattern]                                       #
# first pattern that matches *one* value wins                        #
# ------------------------------------------------------------------ #
RULES = {
    "date":               [ISO_DATE, SLASH_DATE, VACC_DATE, EXCEL_SERIAL, DATE_RE],
    "id":                 [GENERAL_ID],
    "age":                [AGE_PAT],
    "medical_boolean":    [YES_NO],
    "gender":             [GENDER_PAT],
    "outcome":            [OUTCOME_PAT],
    "case_status":        [CASE_STATUS_PAT],
    "genomics_metadata":  [GENOMIC_ID],
    "location":           [LOCATION_PAT],
    "symptoms":           [re.compile(r'[A-Za-z]{3,}.*\|.*')],     # crude pipe-list
    "pre_existing_condition": [
        re.compile(r'^(diab|hyper|neuro|none|preg|yes|no)$', re.IGNORECASE)
    ],
    "contact_setting":    [
        re.compile(r'^(famille|funerailles|communaute|household|school)$', re.IGNORECASE)
    ],
    "transmission":       [
        re.compile(r'^(funerailles|communautaire|nosocomiale)$', re.IGNORECASE)
    ],
    "vaccine_name":       [
        re.compile(r'^(pfizer|moderna|com)$', re.IGNORECASE)
    ],
    "demographic":        [
        re.compile(r'^(male|female|other|age|m|f)$', re.IGNORECASE)
    ],
    # new fields:
    "contact_comment":    [CONTACT_COMMENT_PAT],
    "confirmation_method": [CONFIRMATION_METHOD_PAT],
    "contact_animal":      [CONTACT_ANIMAL_PAT],
    "occupation":          [OCCUPATION_PAT],
    "pathogen":            [PATHOGEN_PAT],
    "source":              [SOURCE_PAT],

}


DELIM_RE = re.compile(r'[|,;/]')

def classify_column(values: Sequence[str]) -> Optional[str]:
    for label, patterns in RULES.items():
        for pat in patterns:
            # look at every raw cell…
            for raw in values:
                if raw is None:
                    continue
                # split into pieces
                for token in DELIM_RE.split(str(raw).strip()):
                    if pat.match(token.strip()):
                        return label
    return None

In [39]:
LABEL_MAP = {
    # Date
    "Vaccination_date": "date",
    "Date_report":"date",
    "Date_onset":  "date",
    "Date_confirmation": "date",
    "Date_of_first_consultation":"date",
    "Date_hospitalisation":  "date",
    "Date_discharge_hospital": "date",
    "Date_admission_ICU":   "date",
    "Date_discharge_ICU":  "date",
    "Date_isolation":  "date",
    "Date_death":  "date",
    "Date_recovered":  "date",
    "Travel_history_entry": "date",
    "Travel_history_start":  "date",
    "Date_entry":  "date",
    "Date_last_modified": "date",

    # ID
    "Contact_ID": "id",
    "ID": "id",

    #Gender
    "Gender": "gender",
    "Sex_at_birth": "gender",
    "Gender_other": "gender",
    "Sex_at_birth_other": "gender",

    #Location
    "Travel_history_location": "location",
    "Location_information": "location",

    # Contact setting
    "Contact_setting": "contact_setting",
    "Contact_setting_other": "contact_setting",

    # demographic
    "Race": "demographic",
    "Ehtnicity": "demographic",

    # Medical Boolean
    "Healthcare_worker": "medical_boolean",
    "Previous_infection": "medical_boolean",
    "Pregnancy_Status": "medical_boolean",
    "Vaccination":  "medical_boolean",
    "Hospitalised":  "medical_boolean",
    "Intensive_care":  "medical_boolean",
    "Home_monitoring":  "medical_boolean",
    "Isolated": "medical_boolean",
    "Contact_with_case": "medical_boolean",
    "Travel_history": "medical_boolean",

    # Sourec
    "Source": "source",
    "Source_II": "source",
    "Source_III": "source",
    "Source_IV": "source",
}

In [40]:
LABEL_MAP_LC = {k.lower(): v.lower() for k, v in LABEL_MAP.items()}

In [41]:
import numpy as np


def remap_labels(arr, mapping=LABEL_MAP_LC):
    """
    • forces each element to lower-case
    • replaces it if the key exists in `mapping`
    • otherwise leaves it as lower-case original
    """
    return np.array([mapping.get(x.lower(), x.lower()) for x in arr])

In [42]:
import pandas as pd

y_train = pd.read_parquet("./labels.parquet").values.flatten()
y_train = remap_labels(y_train)

21

set()

['source', 'contact_comment', 'contact_animal', 'confirmation_method', 'pathogen', 'occupation']


21

In [57]:
# ---------------------------------------------------------------
RAW_CELLS_PARQUET = "./test_data.parquet"
LABELS_PARQUET    = "./test_labels.parquet"

from pathlib import Path
import pandas as pd
import numpy as np
from sklearn.metrics import classification_report, f1_score

# ---------------------------------------------------------------
# 1. load raw cell dataframe  (must contain at least these cols:)
#    • table_name  • column_index  • value
# ---------------------------------------------------------------
df_vals = pd.read_parquet(RAW_CELLS_PARQUET)
df_labels = pd.read_parquet(LABELS_PARQUET)   # same order as df “columns”

value_col  = df_vals.columns[1]            # change if needed
label_col  = df_labels.columns[0]

In [58]:
value_col

'values'

In [59]:
label_col

'type'

In [104]:
# ---------------------------------------------------------------
# 0. paths – edit to match your repo structure
# ---------------------------------------------------------------
RAW_CELLS_PARQUET = "./test_data.parquet"
LABELS_PARQUET    = "./test_labels.parquet"

from pathlib import Path
import pandas as pd
import numpy as np
from sklearn.metrics import classification_report, f1_score

# ---------------------------------------------------------------
# 1. load raw cell dataframe  (must contain at least these cols:)
#    • table_name  • column_index  • value
# ---------------------------------------------------------------
df_vals = pd.read_parquet(RAW_CELLS_PARQUET)
df_labels = pd.read_parquet(LABELS_PARQUET).values.flatten()   # same order as df “columns”

value_col  = df_vals.columns[1]            # change if needed
label_col  = df_labels

values = df_vals[value_col].astype(str).tolist()
y_true = label_col

# optional: same remap/lowercase you used in training
y_true = remap_labels(np.array(y_true))
y_true = np.array([x.lower() for x in y_true])

#values = [v.lower() for v in values]
# if you want case-insensitive match
#print(values)
# ------------------------------------------------------------------ #
# 2. rule-based prediction                                            #
# ------------------------------------------------------------------ #
y_pred = [classify_column([v]) for v in values]   # wrap each val in list
print("Predicted by model")
print(y_pred)
# if classify_column returned None, keep it as special token
y_pred = ["__none__" if p is None else p for p in y_pred]

print("After pred")
#print(y_pred)

print(y_pred)
# ------------------------------------------------------------------ #
# 3. metrics                                                          #
# ------------------------------------------------------------------ #
macro_f1    = f1_score(y_true, y_pred, average="macro", zero_division=0)
weighted_f1 = f1_score(y_true, y_pred, average="weighted", zero_division=0)

print(f"Macro-F1   : {macro_f1:.4f}")
print(f"Weighted F1: {weighted_f1:.4f}")

present_ids   = np.unique(np.concatenate([y_true, y_pred]))
print(classification_report(
    y_true, y_pred,
    labels=present_ids,
    target_names=present_ids,
    digits=3,
    zero_division=0))

Predicted by model
['date', 'medical_boolean', 'age', 'medical_boolean', 'outcome', 'medical_boolean', 'medical_boolean', 'id', 'id', 'location', 'location', 'location', 'location', 'medical_boolean', 'medical_boolean', 'location', 'id', 'medical_boolean', 'medical_boolean', 'age', 'medical_boolean', 'gender', 'location', 'outcome', 'medical_boolean', 'date', 'medical_boolean', 'id', 'medical_boolean', 'date', 'age', 'date', 'gender', 'date', 'medical_boolean', 'date', 'age', 'location', 'medical_boolean', 'gender', 'medical_boolean', 'location', 'date', 'location', 'medical_boolean', 'gender', None, 'date', 'location', 'date']
After pred
['date', 'medical_boolean', 'age', 'medical_boolean', 'outcome', 'medical_boolean', 'medical_boolean', 'id', 'id', 'location', 'location', 'location', 'location', 'medical_boolean', 'medical_boolean', 'location', 'id', 'medical_boolean', 'medical_boolean', 'age', 'medical_boolean', 'gender', 'location', 'outcome', 'medical_boolean', 'date', 'medical_b

In [101]:
tests = ["London", "John Doe", "doctor", "FRANCE", "Laboratoire_123", "2020-15-05", 23, 65]
for t in tests:
    print(t, "→", classify_column([t]))

London → location
John Doe → location
doctor → occupation
FRANCE → location
Laboratoire_123 → None
2020-15-05 → date
23 → age
65 → age
